# TCD stage-supervision config generator

This notebook creates five TCD stage-loss configurations (`uniform`, `geometric_w_st_0.5`, `geometric_w_st_2.0`, `first_only`, and `last_only`) for Decoder RFs 1, 5, 9, 13, and 17 and seeds 1234, 1235, and 1236.

With a kernel size of 3, Decoder RF and refinement steps are related by

\[
\mathrm{RF}_{\mathrm{dec}} = 1 + 2K,
\]

so the refinement-step counts are 0, 2, 4, 6, and 8, respectively.


In [21]:
from __future__ import annotations

import copy
import json
from pathlib import Path
from typing import Any


In [22]:
RAW_CONFIG_ROOT = Path(
    "/home/blue2959/monotonic_tts/exp_configs/v2.3/TCD_stage_mode/raw_configs"
)

# 사용자가 확인한 uniform / Decoder RF=1 / seed=1234 설정을 기준으로 사용
TEMPLATE_DIR = (
    RAW_CONFIG_ROOT
    / "stage_mode=uniform_dec_RF=1_seed=1234"
)

DATA_CONFIG_TEMPLATE_PATH = TEMPLATE_DIR / "data_config.json"
MODEL_CONFIG_TEMPLATE_PATH = TEMPLATE_DIR / "model_config.json"

assert DATA_CONFIG_TEMPLATE_PATH.is_file(), DATA_CONFIG_TEMPLATE_PATH
assert MODEL_CONFIG_TEMPLATE_PATH.is_file(), MODEL_CONFIG_TEMPLATE_PATH

print(f"Raw config root : {RAW_CONFIG_ROOT}")
print(f"Template        : {TEMPLATE_DIR}")


Raw config root : /home/blue2959/monotonic_tts/exp_configs/v2.3/TCD_stage_mode/raw_configs
Template        : /home/blue2959/monotonic_tts/exp_configs/v2.3/TCD_stage_mode/raw_configs/stage_mode=uniform_dec_RF=1_seed=1234


In [23]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def save_json(
    config: dict[str, Any],
    path: Path,
) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            config,
            file,
            ensure_ascii=False,
            indent=4,
        )
        file.write("\n")


data_config_template = load_json(
    DATA_CONFIG_TEMPLATE_PATH
)
model_config_template = load_json(
    MODEL_CONFIG_TEMPLATE_PATH
)

print("Templates loaded.")


Templates loaded.


In [24]:
SEEDS = [1234, 1235, 1236]
DECODER_RFS = [1, 5, 9, 13, 17]

# 실험 폴더에서 사용할 이름 -> model config에 넣을 값
# uniform은 기존 geometric mode에서 w_st=1.0을 사용하면 구현됨.
STAGE_MODE_CONFIGS = {
    "uniform": {
        "coupling_stage_loss_mode": "geometric",
        "coupling_loss_decay_factor": 1.0,
    },
    # "geometric_w_st_0.5": {
    #     "coupling_stage_loss_mode": "geometric",
    #     "coupling_loss_decay_factor": 0.5,
    # },
    "geometric_w_st_2.0": {
        "coupling_stage_loss_mode": "geometric",
        "coupling_loss_decay_factor": 2.0,
    },
    "first_only": {
        "coupling_stage_loss_mode": "first_only",
        "coupling_loss_decay_factor": 1.0,
    },
    "last_only": {
        "coupling_stage_loss_mode": "last_only",
        "coupling_loss_decay_factor": 1.0,
    },
}


DECODER_RF_TO_REFINE_STEPS = {
    1: 0,
    5: 2,
    9: 4,
    13: 6,
    17: 8,
}


assert set(DECODER_RFS) == set(
    DECODER_RF_TO_REFINE_STEPS
)

for decoder_rf in DECODER_RFS:
    num_steps = DECODER_RF_TO_REFINE_STEPS[
        decoder_rf
    ]
    calculated_rf = 1 + 2 * num_steps
    assert calculated_rf == decoder_rf

    print(
        f"Decoder RF={decoder_rf:2d} -> "
        + f"refinement steps={num_steps}"
    )

print()
print("Stage modes:")
for name, values in STAGE_MODE_CONFIGS.items():
    print(f"  {name:10s} -> {values}")


Decoder RF= 1 -> refinement steps=0
Decoder RF= 5 -> refinement steps=2
Decoder RF= 9 -> refinement steps=4
Decoder RF=13 -> refinement steps=6
Decoder RF=17 -> refinement steps=8

Stage modes:
  uniform    -> {'coupling_stage_loss_mode': 'geometric', 'coupling_loss_decay_factor': 1.0}
  geometric_w_st_2.0 -> {'coupling_stage_loss_mode': 'geometric', 'coupling_loss_decay_factor': 2.0}
  first_only -> {'coupling_stage_loss_mode': 'first_only', 'coupling_loss_decay_factor': 1.0}
  last_only  -> {'coupling_stage_loss_mode': 'last_only', 'coupling_loss_decay_factor': 1.0}


In [25]:
# 실험에서 바꾸지 않을 핵심 조건을 template 단계에서 확인
template_nd_aligner = model_config_template["nd_aligner"]
template_txt_enc = template_nd_aligner["txt_enc"]
template_decoder = template_nd_aligner["spec_dec"]
template_aligner = template_nd_aligner["aligner"]

assert template_txt_enc["kernel_sizes"] == [1]
assert template_decoder["decoder_type"] == "coupling"
assert template_decoder["coupling_kernel_size"] == 3
assert template_aligner["unary_network_type"] == "conv"
assert template_aligner["conv_num_layers"] == 6
assert template_aligner["conv_kernel_size"] == [3, 3]

print("Template fixed conditions verified:")
print("  Text RF   = 1")
print("  Scorer RF = 13")
print("  Decoder   = TCD (coupling)")


Template fixed conditions verified:
  Text RF   = 1
  Scorer RF = 13
  Decoder   = TCD (coupling)


In [26]:
def build_data_config(
    *,
    seed: int,
    experiment_name: str,
) -> dict[str, Any]:
    config = copy.deepcopy(data_config_template)

    config["train"]["seed"] = seed
    config["dataset"]["seed"] = seed
    config["extra_exp"]["exp_name"] = experiment_name

    return config


def build_model_config(
    *,
    stage_mode: str,
    decoder_rf: int,
) -> dict[str, Any]:
    if stage_mode not in STAGE_MODE_CONFIGS:
        raise ValueError(
            f"Unsupported stage mode: {stage_mode}"
        )

    if decoder_rf not in DECODER_RF_TO_REFINE_STEPS:
        raise ValueError(
            f"Unsupported Decoder RF: {decoder_rf}"
        )

    config = copy.deepcopy(model_config_template)
    decoder_config = config["nd_aligner"]["spec_dec"]
    mode_config = STAGE_MODE_CONFIGS[stage_mode]

    decoder_config["decoder_type"] = "coupling"
    decoder_config["coupling_kernel_size"] = 3
    decoder_config["coupling_num_refine_steps"] = (
        DECODER_RF_TO_REFINE_STEPS[decoder_rf]
    )
    decoder_config["coupling_stage_loss_mode"] = (
        mode_config["coupling_stage_loss_mode"]
    )
    decoder_config["coupling_loss_decay_factor"] = (
        mode_config["coupling_loss_decay_factor"]
    )

    decoder_config[
        "coupling_normalize_loss_weights"
    ] = True

    return config


In [27]:
experiment_specs = []

for stage_mode in STAGE_MODE_CONFIGS:
    for decoder_rf in DECODER_RFS:
        for seed in SEEDS:
            experiment_name = (
                f"stage_mode={stage_mode}"
                f"_dec_RF={decoder_rf}"
                f"_seed={seed}"
            )

            experiment_specs.append(
                {
                    "stage_mode": stage_mode,
                    "decoder_rf": decoder_rf,
                    "seed": seed,
                    "experiment_name": experiment_name,
                }
            )

expected_num_experiments = (
    len(STAGE_MODE_CONFIGS)
    * len(DECODER_RFS)
    * len(SEEDS)
)

assert len(experiment_specs) == expected_num_experiments

print(
    f"Number of experiments: "
    + f"{len(experiment_specs)}"
)

for spec in experiment_specs:
    print(spec["experiment_name"])


Number of experiments: 60
stage_mode=uniform_dec_RF=1_seed=1234
stage_mode=uniform_dec_RF=1_seed=1235
stage_mode=uniform_dec_RF=1_seed=1236
stage_mode=uniform_dec_RF=5_seed=1234
stage_mode=uniform_dec_RF=5_seed=1235
stage_mode=uniform_dec_RF=5_seed=1236
stage_mode=uniform_dec_RF=9_seed=1234
stage_mode=uniform_dec_RF=9_seed=1235
stage_mode=uniform_dec_RF=9_seed=1236
stage_mode=uniform_dec_RF=13_seed=1234
stage_mode=uniform_dec_RF=13_seed=1235
stage_mode=uniform_dec_RF=13_seed=1236
stage_mode=uniform_dec_RF=17_seed=1234
stage_mode=uniform_dec_RF=17_seed=1235
stage_mode=uniform_dec_RF=17_seed=1236
stage_mode=geometric_w_st_2.0_dec_RF=1_seed=1234
stage_mode=geometric_w_st_2.0_dec_RF=1_seed=1235
stage_mode=geometric_w_st_2.0_dec_RF=1_seed=1236
stage_mode=geometric_w_st_2.0_dec_RF=5_seed=1234
stage_mode=geometric_w_st_2.0_dec_RF=5_seed=1235
stage_mode=geometric_w_st_2.0_dec_RF=5_seed=1236
stage_mode=geometric_w_st_2.0_dec_RF=9_seed=1234
stage_mode=geometric_w_st_2.0_dec_RF=9_seed=1235
stage_

In [28]:
created_experiments = []

for spec in experiment_specs:
    stage_mode = spec["stage_mode"]
    decoder_rf = spec["decoder_rf"]
    seed = spec["seed"]
    experiment_name = spec["experiment_name"]

    experiment_dir = RAW_CONFIG_ROOT / experiment_name
    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    data_config = build_data_config(
        seed=seed,
        experiment_name=experiment_name,
    )
    model_config = build_model_config(
        stage_mode=stage_mode,
        decoder_rf=decoder_rf,
    )

    save_json(
        data_config,
        experiment_dir / "data_config.json",
    )
    save_json(
        model_config,
        experiment_dir / "model_config.json",
    )

    created_experiments.append(experiment_dir)

print(
    f"Created/updated {len(created_experiments)} "
    + "experiment directories."
)


Created/updated 60 experiment directories.


In [29]:
for spec in experiment_specs:
    stage_mode = spec["stage_mode"]
    expected_decoder_rf = spec["decoder_rf"]
    expected_seed = spec["seed"]
    experiment_name = spec["experiment_name"]

    experiment_dir = RAW_CONFIG_ROOT / experiment_name
    data_config_path = experiment_dir / "data_config.json"
    model_config_path = experiment_dir / "model_config.json"

    assert data_config_path.is_file()
    assert model_config_path.is_file()

    data_config = load_json(data_config_path)
    model_config = load_json(model_config_path)

    nd_aligner_config = model_config["nd_aligner"]
    txt_enc_config = nd_aligner_config["txt_enc"]
    decoder_config = nd_aligner_config["spec_dec"]
    aligner_config = nd_aligner_config["aligner"]
    expected_mode_config = STAGE_MODE_CONFIGS[
        stage_mode
    ]

    # Data config: 이름과 두 seed 위치를 모두 검증
    assert (
        data_config["extra_exp"]["exp_name"]
        == experiment_name
    )
    assert data_config["train"]["seed"] == expected_seed
    assert data_config["dataset"]["seed"] == expected_seed

    # 고정 조건: Text RF=1, Scorer RF=13, TCD
    assert txt_enc_config["kernel_sizes"] == [1]
    assert aligner_config["unary_network_type"] == "conv"
    assert aligner_config["conv_num_layers"] == 6
    assert aligner_config["conv_kernel_size"] == [3, 3]
    assert decoder_config["decoder_type"] == "coupling"
    assert decoder_config["coupling_kernel_size"] == 3

    # 변경 조건: Decoder RF와 stage supervision mode
    assert (
        decoder_config["coupling_num_refine_steps"]
        == DECODER_RF_TO_REFINE_STEPS[
            expected_decoder_rf
        ]
    )
    assert (
        decoder_config["coupling_stage_loss_mode"]
        == expected_mode_config[
            "coupling_stage_loss_mode"
        ]
    )
    assert (
        decoder_config["coupling_loss_decay_factor"]
        == expected_mode_config[
            "coupling_loss_decay_factor"
        ]
    )
    assert (
        decoder_config[
            "coupling_normalize_loss_weights"
        ]
        is True
    )

print(
    f"All {len(experiment_specs)} configurations "
    + "verified successfully."
)


All 60 configurations verified successfully.


## Compute-saving note

When Decoder RF is 1, the number of refinement steps is zero. Therefore, all five stage modes supervise the same single initial output and are mathematically equivalent. The notebook still creates all five RF=1 configurations to keep the sweep rectangular, but their training results can be reused if desired.
